# Quantization Deep Dive

This notebook upgrades the earlier self-contained draft to use the real helpers in `llm_from_scratch.quantization`.

## 1. Why quantization matters for LLM memory and inference

Large language models spend most of their memory budget on parameter tensors, activations in flight, and the KV cache used during autoregressive decoding. Quantization reduces the bytes needed to store those values, which can lower memory pressure and sometimes improve effective throughput when kernels support lower-precision math efficiently.

## 2. Floating point versus integer ranges

Floating-point tensors cover wide dynamic ranges, while low-bit integer tensors provide only a small number of discrete levels. Quantization works by choosing a scale and zero-point so that the integer grid approximates the floating-point values we care about.

In [ ]:
import torch

from llm_from_scratch.quantization import (
    dequantize_tensor,
    estimate_kv_cache_bytes,
    fake_quantize_tensor,
    quantization_error,
    quantize_per_channel,
    quantize_tensor,
)

float_values = torch.tensor([-2.0, -0.5, 0.0, 0.75, 2.0])
q4, params4 = quantize_tensor(float_values, num_bits=4, symmetric=True)
q8, params8 = quantize_tensor(float_values, num_bits=8, symmetric=False)
q4, params4, q8, params8

## 3. Derivation of scale and zero-point

For affine quantization, the core formulas are `q = round(x / scale + zero_point)` and `x_hat = scale * (q - zero_point)`. Asymmetric quantization uses the observed minimum and maximum to fit values into an unsigned interval. Symmetric quantization centers the integer range around zero and fixes `zero_point = 0`.

## 4. Symmetric versus asymmetric quantization

Symmetric quantization is common for weights because many kernels assume signed integer ranges with a zero-point of zero. Asymmetric quantization can better fit activations or other tensors whose ranges are shifted away from zero. The tradeoff is a slightly more complex mapping.

## 5. Per-tensor, per-channel, and groupwise intuition

Per-tensor quantization uses one scale for the entire tensor. Per-channel quantization gives each channel its own scale, which often reduces error when channels have very different ranges. Groupwise quantization sits between the two: several neighboring channels share a scale, reducing metadata while keeping tighter ranges than pure per-tensor quantization.

In [ ]:
weights = torch.randn(4, 8)
q, params = quantize_tensor(weights, num_bits=4, symmetric=True)
weights_hat = dequantize_tensor(q, params)
q_channel, channel_params = quantize_per_channel(weights, axis=0, num_bits=4)
per_channel_recon = torch.stack([
    dequantize_tensor(q_channel[i], channel_params[i])
    for i in range(q_channel.shape[0])
])
kv_bytes_fp16 = estimate_kv_cache_bytes(2, 4, 16, 128, 1, bytes_per_value=2)
params, quantization_error(weights, weights_hat), q_channel.shape, len(channel_params), kv_bytes_fp16

## 6. From-scratch quantize/dequantize demo using `quantize_tensor`

The helpers in `llm_from_scratch.quantization` are deliberately small: `quantize_tensor` returns integer values plus `QuantizationParams`, and `dequantize_tensor` maps back to a floating-point approximation. This makes the approximation error visible instead of hiding it behind a framework abstraction.

## 7. Quantization error measurement on a toy weight matrix

We can compare reconstruction error numerically. Lower bit-widths generally increase error, while per-channel quantization often improves it when rows or columns have different dynamic ranges.

In [ ]:
toy = torch.tensor([[-2.0, -0.5, 0.5, 2.0], [0.1, 0.2, 0.3, 0.4]], dtype=torch.float32)
toy_8_q, toy_8_params = quantize_tensor(toy, num_bits=8, symmetric=True)
toy_4_q, toy_4_params = quantize_tensor(toy, num_bits=4, symmetric=True)
toy_8_hat = dequantize_tensor(toy_8_q, toy_8_params)
toy_4_hat = dequantize_tensor(toy_4_q, toy_4_params)
error_8bit = quantization_error(toy, toy_8_hat)
error_4bit = quantization_error(toy, toy_4_hat)
error_8bit, error_4bit

## 8. Weight-only versus activation quantization

Weight-only quantization is usually the easiest first win for LLM deployment because parameter tensors dominate model storage and do not depend on runtime input statistics. Activation quantization can reduce bandwidth further, but it introduces extra calibration and kernel constraints because activations change from token to token.

## 9. KV-cache memory discussion

The KV cache stores keys and values for every generated token, layer, head, and batch element. That makes its memory cost grow linearly with sequence length, which is why long-context decoding can become memory-bound even when model weights are already quantized.

In [ ]:
per_tensor_err = quantization_error(weights, weights_hat)
per_channel_err = quantization_error(weights, per_channel_recon)
fake_quantized = fake_quantize_tensor(weights, num_bits=4, symmetric=True)
kv_bytes_int8 = estimate_kv_cache_bytes(2, 4, 16, 128, 1, bytes_per_value=1)
{
    'per_tensor_4bit': per_tensor_err,
    'per_channel_4bit': per_channel_err,
    'fake_quantized_dtype': str(fake_quantized.dtype),
    'kv_bytes_fp16': kv_bytes_fp16,
    'kv_bytes_int8': kv_bytes_int8,
}

## 10. KV-cache memory estimate with `estimate_kv_cache_bytes`

`estimate_kv_cache_bytes` makes the scaling law explicit: keys and values are both stored, so the estimate multiplies by two before accounting for layers, heads, head dimension, sequence length, batch size, and bytes per stored value. Reducing bytes per value directly reduces this footprint.

## 11. Current PyTorch and Hugging Face API map

- **`torchao`**: PyTorch's quantization work for modern inference and training experiments. Treat it as the first place to look for native PyTorch low-precision utilities and evolving recipes.
- **bitsandbytes-style loading**: Hugging Face loaders commonly expose `load_in_8bit=True` or 4-bit `BitsAndBytesConfig` settings for weight-only quantization flows.
- **GPTQ**: post-training quantization that fits quantized weights using calibration data and second-order approximations.
- **AWQ**: activation-aware weight quantization that tries to preserve important channels by using calibration activations.

These APIs sit above the teaching helpers in this project. The helpers here are for understanding the math and memory tradeoffs before reaching for production kernels.

## 12. Apple Silicon, CUDA, and CPU caveats

Quantization is not just about data types; it is also about kernels. CUDA typically has the broadest coverage for optimized low-bit kernels. CPU backends may support some integer paths but not all LLM-specific fused kernels. Apple Silicon can run float16 and int8 workloads well in some settings, but library support for specific 4-bit and fused attention paths often lags CUDA-focused stacks.

## 13. Exercises and solution references

Continue with [exercises/quantization.md](../exercises/quantization.md) and check worked answers in [exercises/solutions/quantization_solutions.md](../exercises/solutions/quantization_solutions.md). For the math derivation, see [docs/math/quantization.md](../docs/math/quantization.md).